# Arquitectura Empresarial para ML a Escala

**Objetivo**: Diseñar plataformas de ML empresariales que soporten múltiples equipos, modelos en producción, y gobernanza a gran escala.

---

## 1. ML Platform: visión empresarial

Una **ML Platform** empresarial debe soportar todo el ciclo de vida de ML para múltiples equipos:

```
┌────────────────────────────────────────────────────────────────────┐
│                    ML PLATFORM LAYERS                              │
├────────────────────────────────────────────────────────────────────┤
│  Developer Experience Layer                                        │
│  • Notebooks • IDEs • CLI • APIs • Self-service portal            │
├────────────────────────────────────────────────────────────────────┤
│  ML Workflow Layer                                                 │
│  • Experimentation • Training • Deployment • Monitoring           │
├────────────────────────────────────────────────────────────────────┤
│  ML Infrastructure Layer                                           │
│  • Feature Store • Model Registry • Compute • Storage             │
├────────────────────────────────────────────────────────────────────┤
│  Data Infrastructure Layer                                         │
│  • Data Lake • Data Warehouse • Streaming • Catalog               │
├────────────────────────────────────────────────────────────────────┤
│  Platform Services Layer                                           │
│  • IAM • Monitoring • Logging • Cost management • Governance      │
└────────────────────────────────────────────────────────────────────┘
```

### Principios de diseño

| Principio | Descripción | Beneficio |
|-----------|-------------|------------|
| **Self-service** | Equipos pueden experimentar sin intervención de plataforma | Velocidad de innovación |
| **Abstraído** | Complejidad de infraestructura oculta | Enfoque en ML, no en DevOps |
| **Estandarizado** | Patrones comunes reutilizables | Consistencia y calidad |
| **Observable** | Visibilidad completa del ciclo de vida | Debugging y optimización |
| **Gobernado** | Control centralizado de políticas | Compliance y seguridad |

## 2. Model Registry y versionado

### 2.1 Model Registry empresarial

Un Model Registry a escala debe manejar:
- **Cientos/miles de modelos** de múltiples equipos
- **Múltiples versiones** por modelo con historial completo
- **Metadatos enriquecidos**: métricas, linaje, aprobaciones
- **Estados del modelo**: development → staging → production → archived

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime
from enum import Enum

class ModelStage(Enum):
    DEVELOPMENT = "development"
    STAGING = "staging"
    PRODUCTION = "production"
    ARCHIVED = "archived"

@dataclass
class ModelMetadata:
    """Metadatos completos de un modelo"""
    model_id: str
    version: str
    stage: ModelStage
    
    # Linaje
    training_data_version: str
    feature_store_snapshot: str
    code_commit_hash: str
    
    # Métricas
    metrics: Dict[str, float]
    
    # Metadatos operacionales
    framework: str  # tensorflow, pytorch, sklearn, etc.
    model_size_mb: float
    inference_latency_p99_ms: float
    
    # Gobernanza
    owner_team: str
    approved_by: Optional[str] = None
    approval_date: Optional[datetime] = None
    
    # Auditoría
    created_at: datetime = field(default_factory=datetime.now)
    deployed_at: Optional[datetime] = None

class EnterpriseModelRegistry:
    """Model Registry empresarial"""
    
    def __init__(self):
        self.models: Dict[str, List[ModelMetadata]] = {}
        self.production_models: Dict[str, ModelMetadata] = {}
    
    def register_model(self, metadata: ModelMetadata):
        """Registra nueva versión de modelo"""
        if metadata.model_id not in self.models:
            self.models[metadata.model_id] = []
        
        self.models[metadata.model_id].append(metadata)
        print(f"✓ Modelo registrado: {metadata.model_id} v{metadata.version}")
        print(f"  Team: {metadata.owner_team}")
        print(f"  Stage: {metadata.stage.value}")
        print(f"  Métricas: {metadata.metrics}")
    
    def promote_to_production(self, model_id: str, version: str, 
                             approved_by: str) -> bool:
        """Promueve modelo a producción"""
        # Encontrar modelo
        model = self._find_model(model_id, version)
        if not model:
            print(f"❌ Modelo no encontrado: {model_id} v{version}")
            return False
        
        # Validaciones de gobernanza
        if not self._validate_for_production(model):
            return False
        
        # Archivar modelo anterior en producción
        if model_id in self.production_models:
            old_model = self.production_models[model_id]
            old_model.stage = ModelStage.ARCHIVED
            print(f"  → Modelo anterior archivado: v{old_model.version}")
        
        # Promover nuevo modelo
        model.stage = ModelStage.PRODUCTION
        model.approved_by = approved_by
        model.approval_date = datetime.now()
        model.deployed_at = datetime.now()
        self.production_models[model_id] = model
        
        print(f"✅ Modelo promovido a PRODUCTION: {model_id} v{version}")
        print(f"   Aprobado por: {approved_by}")
        return True
    
    def get_production_model(self, model_id: str) -> Optional[ModelMetadata]:
        """Obtiene modelo en producción"""
        return self.production_models.get(model_id)
    
    def get_model_lineage(self, model_id: str, version: str) -> Dict:
        """Obtiene linaje completo del modelo"""
        model = self._find_model(model_id, version)
        if not model:
            return {}
        
        return {
            'model': f"{model_id} v{version}",
            'training_data': model.training_data_version,
            'features': model.feature_store_snapshot,
            'code': model.code_commit_hash,
            'created': model.created_at.isoformat(),
            'owner': model.owner_team
        }
    
    def list_models_by_team(self, team: str) -> List[str]:
        """Lista modelos de un equipo"""
        team_models = set()
        for model_id, versions in self.models.items():
            if any(v.owner_team == team for v in versions):
                team_models.add(model_id)
        return sorted(team_models)
    
    def _find_model(self, model_id: str, version: str) -> Optional[ModelMetadata]:
        """Busca modelo específico"""
        if model_id not in self.models:
            return None
        for model in self.models[model_id]:
            if model.version == version:
                return model
        return None
    
    def _validate_for_production(self, model: ModelMetadata) -> bool:
        """Validaciones antes de promover a producción"""
        validations = []
        
        # 1. Latencia aceptable (<100ms)
        if model.inference_latency_p99_ms > 100:
            validations.append(f"❌ Latencia muy alta: {model.inference_latency_p99_ms}ms")
        else:
            validations.append("✓ Latencia OK")
        
        # 2. Métricas mínimas
        if 'accuracy' in model.metrics and model.metrics['accuracy'] < 0.8:
            validations.append(f"❌ Accuracy baja: {model.metrics['accuracy']}")
        else:
            validations.append("✓ Métricas OK")
        
        # 3. Tamaño razonable (<500MB)
        if model.model_size_mb > 500:
            validations.append(f"⚠️  Modelo grande: {model.model_size_mb}MB")
        else:
            validations.append("✓ Tamaño OK")
        
        for v in validations:
            print(f"  {v}")
        
        # Aprobar si no hay errores críticos
        return not any('❌' in v for v in validations)

# Demo
print("=" * 70)
print("DEMO: Enterprise Model Registry")
print("=" * 70)

registry = EnterpriseModelRegistry()

# Registrar modelo v1
print("\n📝 Registrando modelo fraud_detection v1.0\n")
model_v1 = ModelMetadata(
    model_id="fraud_detection",
    version="1.0",
    stage=ModelStage.DEVELOPMENT,
    training_data_version="dataset_v2.1",
    feature_store_snapshot="features_2024-11-01",
    code_commit_hash="a3f5b2c",
    metrics={'accuracy': 0.85, 'precision': 0.82, 'recall': 0.88},
    framework="pytorch",
    model_size_mb=120.5,
    inference_latency_p99_ms=45.0,
    owner_team="fraud_team"
)
registry.register_model(model_v1)

# Promover a producción
print("\n🚀 Promoviendo a producción\n")
registry.promote_to_production("fraud_detection", "1.0", "senior_ml_engineer")

# Registrar modelo v2 mejorado
print("\n📝 Registrando modelo fraud_detection v2.0\n")
model_v2 = ModelMetadata(
    model_id="fraud_detection",
    version="2.0",
    stage=ModelStage.STAGING,
    training_data_version="dataset_v2.3",
    feature_store_snapshot="features_2024-11-07",
    code_commit_hash="d9e8f1a",
    metrics={'accuracy': 0.92, 'precision': 0.90, 'recall': 0.94},
    framework="pytorch",
    model_size_mb=135.2,
    inference_latency_p99_ms=52.0,
    owner_team="fraud_team"
)
registry.register_model(model_v2)

# Promover v2
print("\n🚀 Promoviendo v2.0 a producción\n")
registry.promote_to_production("fraud_detection", "2.0", "ml_lead")

# Consultar linaje
print("\n🔍 Linaje del modelo en producción\n")
import json
lineage = registry.get_model_lineage("fraud_detection", "2.0")
print(json.dumps(lineage, indent=2))

## 3. Arquitectura multi-modelo

### Estrategias de deployment

| Estrategia | Caso de uso | Implementación |
|------------|-------------|----------------|
| **Single model** | MVP, modelo único | 1 endpoint, 1 versión |
| **Blue-Green** | Cambios seguros | 2 ambientes, switch instantáneo |
| **Canary** | Rollout gradual | % de tráfico al nuevo modelo |
| **A/B Testing** | Experimentación | Split de tráfico aleatorio |
| **Shadow mode** | Validación sin impacto | Modelo nuevo sin afectar respuesta |
| **Multi-armed bandit** | Optimización continua | Asignación dinámica según performance |

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
import random

@dataclass
class ModelEndpoint:
    """Representa un modelo deployado"""
    model_id: str
    version: str
    endpoint_url: str
    traffic_percentage: float
    is_shadow: bool = False

class MultiModelRouter:
    """Router para múltiples modelos con diferentes estrategias"""
    
    def __init__(self):
        self.endpoints: Dict[str, List[ModelEndpoint]] = {}
        self.request_count = 0
    
    def register_endpoint(self, model_id: str, endpoint: ModelEndpoint):
        """Registra endpoint de modelo"""
        if model_id not in self.endpoints:
            self.endpoints[model_id] = []
        self.endpoints[model_id].append(endpoint)
        print(f"✓ Endpoint registrado: {model_id} v{endpoint.version}")
        print(f"  Traffic: {endpoint.traffic_percentage}%")
        print(f"  Shadow: {endpoint.is_shadow}")
    
    def route_request(self, model_id: str, request_data: Dict) -> Dict:
        """Rutea request según estrategia configurada"""
        self.request_count += 1
        
        if model_id not in self.endpoints:
            return {'error': 'Model not found'}
        
        endpoints = self.endpoints[model_id]
        
        # Seleccionar endpoint según distribución de tráfico
        selected_endpoint = self._select_endpoint(endpoints)
        
        # Invocar modelo principal
        response = self._invoke_model(selected_endpoint, request_data)
        
        # Invocar modelos en shadow mode (sin retornar resultado)
        shadow_endpoints = [e for e in endpoints if e.is_shadow]
        for shadow_ep in shadow_endpoints:
            self._invoke_model(shadow_ep, request_data, is_shadow=True)
        
        return response
    
    def _select_endpoint(self, endpoints: List[ModelEndpoint]) -> ModelEndpoint:
        """Selecciona endpoint según traffic percentage"""
        # Filtrar solo no-shadow
        active_endpoints = [e for e in endpoints if not e.is_shadow]
        
        # Selección ponderada
        rand = random.uniform(0, 100)
        cumulative = 0
        for ep in active_endpoints:
            cumulative += ep.traffic_percentage
            if rand <= cumulative:
                return ep
        
        return active_endpoints[0]  # fallback
    
    def _invoke_model(self, endpoint: ModelEndpoint, request_data: Dict, 
                     is_shadow: bool = False) -> Dict:
        """Simula invocación de modelo"""
        # En producción: HTTP call al endpoint
        prediction = random.choice([0, 1])  # fraud or not
        confidence = random.uniform(0.7, 0.99)
        
        mode = "SHADOW" if is_shadow else "ACTIVE"
        print(f"  [{mode}] {endpoint.model_id} v{endpoint.version}: "
              f"prediction={prediction}, confidence={confidence:.2f}")
        
        return {
            'prediction': prediction,
            'confidence': confidence,
            'model_version': endpoint.version
        }
    
    def configure_ab_test(self, model_id: str, version_a: str, version_b: str,
                         traffic_split: float = 0.5):
        """Configura A/B test entre dos versiones"""
        print(f"\n🔬 Configurando A/B test: {model_id}")
        print(f"   Version A ({version_a}): {traffic_split*100}%")
        print(f"   Version B ({version_b}): {(1-traffic_split)*100}%")
        
        # Limpiar endpoints anteriores
        self.endpoints[model_id] = []
        
        # Registrar ambas versiones
        self.register_endpoint(model_id, ModelEndpoint(
            model_id=model_id,
            version=version_a,
            endpoint_url=f"http://api/v{version_a}",
            traffic_percentage=traffic_split * 100
        ))
        
        self.register_endpoint(model_id, ModelEndpoint(
            model_id=model_id,
            version=version_b,
            endpoint_url=f"http://api/v{version_b}",
            traffic_percentage=(1 - traffic_split) * 100
        ))
    
    def configure_canary(self, model_id: str, stable_version: str, 
                        canary_version: str, canary_percentage: float = 0.1):
        """Configura canary deployment"""
        print(f"\n🐤 Configurando Canary deployment: {model_id}")
        print(f"   Stable ({stable_version}): {(1-canary_percentage)*100}%")
        print(f"   Canary ({canary_version}): {canary_percentage*100}%")
        
        self.endpoints[model_id] = []
        
        self.register_endpoint(model_id, ModelEndpoint(
            model_id=model_id,
            version=stable_version,
            endpoint_url=f"http://api/v{stable_version}",
            traffic_percentage=(1 - canary_percentage) * 100
        ))
        
        self.register_endpoint(model_id, ModelEndpoint(
            model_id=model_id,
            version=canary_version,
            endpoint_url=f"http://api/v{canary_version}",
            traffic_percentage=canary_percentage * 100
        ))

# Demo
print("=" * 70)
print("DEMO: Multi-Model Routing Strategies")
print("=" * 70)

router = MultiModelRouter()

# Escenario 1: A/B Testing
router.configure_ab_test("fraud_detection", "2.0", "3.0", traffic_split=0.5)

print("\n--- Simulando 5 requests con A/B test ---")
for i in range(5):
    print(f"\nRequest {i+1}:")
    router.route_request("fraud_detection", {'transaction_id': i})

# Escenario 2: Canary deployment
router.configure_canary("fraud_detection", "2.0", "3.0", canary_percentage=0.1)

print("\n--- Simulando 5 requests con Canary (10%) ---")
for i in range(5):
    print(f"\nRequest {i+1}:")
    router.route_request("fraud_detection", {'transaction_id': i})

# Escenario 3: Shadow mode
print("\n🌑 Configurando Shadow mode\n")
router.endpoints["fraud_detection"] = []
router.register_endpoint("fraud_detection", ModelEndpoint(
    model_id="fraud_detection",
    version="2.0",
    endpoint_url="http://api/v2.0",
    traffic_percentage=100.0
))
router.register_endpoint("fraud_detection", ModelEndpoint(
    model_id="fraud_detection",
    version="3.0",
    endpoint_url="http://api/v3.0",
    traffic_percentage=0.0,
    is_shadow=True
))

print("\n--- Request con Shadow mode ---")
print("\nRequest:")
router.route_request("fraud_detection", {'transaction_id': 999})

## 4. Taller: Sistema de mantenimiento predictivo

### 4.1 Caso de negocio

**Empresa**: Manufacturera con 500 máquinas industriales
**Problema**: Fallas inesperadas causan pérdidas de $50k/hora
**Solución**: Mantenimiento predictivo con ML

### 4.2 Arquitectura propuesta

In [ ]:
from dataclasses import dataclass
from typing import List, Dict
from datetime import datetime, timedelta
import random

@dataclass
class MachineSensor:
    """Datos de sensores de máquina"""
    machine_id: str
    timestamp: datetime
    temperature: float
    vibration: float
    pressure: float
    rpm: float

@dataclass
class MaintenanceEvent:
    """Evento de mantenimiento"""
    machine_id: str
    timestamp: datetime
    event_type: str  # preventive, corrective, failure
    cost: float

class PredictiveMaintenanceArchitecture:
    """Arquitectura completa para mantenimiento predictivo"""
    
    def __init__(self):
        # Simulación de componentes
        self.feature_store = {}
        self.model_registry = {}
        self.predictions = {}
        self.maintenance_schedule = []
        
        # KPIs
        self.total_cost_saved = 0
        self.failures_prevented = 0
    
    def ingest_sensor_data(self, sensor_data: MachineSensor):
        """1. Ingesta de datos de sensores (streaming)"""
        # En producción: Kafka → Flink/Spark Streaming
        pass
    
    def compute_features(self, machine_id: str, 
                        sensor_history: List[MachineSensor]) -> Dict:
        """2. Feature engineering sobre ventana temporal"""
        if not sensor_history:
            return {}
        
        # Agregaciones sobre ventana de 24 horas
        temps = [s.temperature for s in sensor_history]
        vibrations = [s.vibration for s in sensor_history]
        
        features = {
            'temp_mean_24h': sum(temps) / len(temps),
            'temp_max_24h': max(temps),
            'temp_std_24h': self._std(temps),
            'vibration_mean_24h': sum(vibrations) / len(vibrations),
            'vibration_max_24h': max(vibrations),
            'vibration_trend': self._trend(vibrations),
            'hours_since_last_maintenance': random.uniform(100, 500)
        }
        
        return features
    
    def predict_failure_probability(self, machine_id: str, 
                                    features: Dict) -> float:
        """3. Predicción de probabilidad de falla"""
        # En producción: modelo entrenado (Random Forest, XGBoost, etc.)
        
        # Simulación basada en features
        risk_score = 0.0
        
        # Alta temperatura → mayor riesgo
        if features.get('temp_max_24h', 0) > 85:
            risk_score += 0.3
        
        # Alta vibración → mayor riesgo
        if features.get('vibration_max_24h', 0) > 0.8:
            risk_score += 0.3
        
        # Tendencia creciente en vibración
        if features.get('vibration_trend', 0) > 0.1:
            risk_score += 0.2
        
        # Tiempo desde mantenimiento
        if features.get('hours_since_last_maintenance', 0) > 400:
            risk_score += 0.2
        
        return min(risk_score, 0.95)
    
    def schedule_maintenance(self, machine_id: str, failure_probability: float,
                           features: Dict) -> Dict:
        """4. Decisión de mantenimiento"""
        recommendation = {
            'machine_id': machine_id,
            'failure_probability': failure_probability,
            'action': 'none',
            'priority': 'low',
            'estimated_cost_if_failure': 50000,  # $50k/hora downtime
            'maintenance_cost': 2000  # $2k mantenimiento preventivo
        }
        
        if failure_probability > 0.7:
            recommendation['action'] = 'immediate_maintenance'
            recommendation['priority'] = 'critical'
            self.maintenance_schedule.append(recommendation)
            self.failures_prevented += 1
            self.total_cost_saved += 48000  # $50k - $2k
        elif failure_probability > 0.5:
            recommendation['action'] = 'schedule_within_week'
            recommendation['priority'] = 'high'
            self.maintenance_schedule.append(recommendation)
        elif failure_probability > 0.3:
            recommendation['action'] = 'monitor_closely'
            recommendation['priority'] = 'medium'
        
        return recommendation
    
    def run_assessment(self, num_machines: int = 10):
        """Ejecuta evaluación completa"""
        print("=" * 70)
        print(f"TALLER: Evaluación de Sistema de Mantenimiento Predictivo")
        print(f"Máquinas evaluadas: {num_machines}")
        print("=" * 70)
        
        for i in range(num_machines):
            machine_id = f"machine_{i+1:03d}"
            
            # Generar datos de sensores simulados
            sensor_history = [
                MachineSensor(
                    machine_id=machine_id,
                    timestamp=datetime.now() - timedelta(hours=h),
                    temperature=random.uniform(70, 95),
                    vibration=random.uniform(0.3, 1.0),
                    pressure=random.uniform(80, 120),
                    rpm=random.uniform(1000, 2000)
                )
                for h in range(24)
            ]
            
            # Pipeline completo
            features = self.compute_features(machine_id, sensor_history)
            failure_prob = self.predict_failure_probability(machine_id, features)
            recommendation = self.schedule_maintenance(machine_id, failure_prob, features)
            
            if recommendation['action'] != 'none':
                print(f"\n{machine_id}:")
                print(f"  Probabilidad falla: {failure_prob:.1%}")
                print(f"  Acción: {recommendation['action']}")
                print(f"  Prioridad: {recommendation['priority']}")
        
        # Resumen de KPIs
        print("\n" + "=" * 70)
        print("📊 RESULTADOS DEL SISTEMA")
        print("=" * 70)
        print(f"\n✅ Fallas previstas y evitadas: {self.failures_prevented}")
        print(f"💰 Costo ahorrado estimado: ${self.total_cost_saved:,}")
        print(f"📅 Mantenimientos programados: {len(self.maintenance_schedule)}")
        
        # ROI
        system_cost = 100000  # $100k inversión en sistema ML
        roi = ((self.total_cost_saved - system_cost) / system_cost) * 100
        print(f"\n📈 ROI del sistema: {roi:.0f}%")
        
        if roi > 200:
            print("\n🎯 RECOMENDACIÓN: Expandir a todas las máquinas")
        elif roi > 100:
            print("\n✓ RECOMENDACIÓN: Continuar con el sistema")
        else:
            print("\n⚠️  RECOMENDACIÓN: Optimizar modelo y features")
    
    def _std(self, values: List[float]) -> float:
        """Desviación estándar"""
        mean = sum(values) / len(values)
        variance = sum((x - mean) ** 2 for x in values) / len(values)
        return variance ** 0.5
    
    def _trend(self, values: List[float]) -> float:
        """Tendencia (slope simplificado)"""
        if len(values) < 2:
            return 0.0
        return (values[-1] - values[0]) / len(values)

# Ejecutar taller
architecture = PredictiveMaintenanceArchitecture()
architecture.run_assessment(num_machines=20)

## 5. Observabilidad de ML en producción

### 5.1 Capas de observabilidad

```
┌─────────────────────────────────────────────────────────┐
│ BUSINESS METRICS                                        │
│ • Conversión • Revenue • User satisfaction              │
├─────────────────────────────────────────────────────────┤
│ MODEL METRICS                                           │
│ • Accuracy • Precision • Recall • AUC                   │
├─────────────────────────────────────────────────────────┤
│ DATA QUALITY METRICS                                    │
│ • Data drift • Schema validation • Completeness         │
├─────────────────────────────────────────────────────────┤
│ OPERATIONAL METRICS                                     │
│ • Latency • Throughput • Error rate • Resource usage   │
└─────────────────────────────────────────────────────────┘
```

### 5.2 Data Drift Detection

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
import random

@dataclass
class DriftReport:
    feature_name: str
    drift_score: float
    is_drifted: bool
    threshold: float = 0.3

class MLObservability:
    """Sistema de observabilidad para modelos en producción"""
    
    def __init__(self):
        # Baseline: distribuciones de entrenamiento
        self.training_distributions = {}
        
        # Métricas actuales
        self.current_metrics = {
            'requests_per_second': 0,
            'latency_p50_ms': 0,
            'latency_p99_ms': 0,
            'error_rate': 0,
            'model_accuracy': 0
        }
    
    def set_training_baseline(self, feature_distributions: Dict[str, Dict]):
        """Establece baseline de entrenamiento"""
        self.training_distributions = feature_distributions
        print("✓ Baseline de entrenamiento establecido")
        print(f"  Features monitoreadas: {list(feature_distributions.keys())}")
    
    def detect_data_drift(self, production_data: Dict[str, List[float]]) -> List[DriftReport]:
        """Detecta drift en datos de producción"""
        drift_reports = []
        
        for feature_name, prod_values in production_data.items():
            if feature_name not in self.training_distributions:
                continue
            
            # Calcular drift (KS test simplificado)
            training_dist = self.training_distributions[feature_name]
            drift_score = self._calculate_drift(training_dist, prod_values)
            
            report = DriftReport(
                feature_name=feature_name,
                drift_score=drift_score,
                is_drifted=drift_score > 0.3
            )
            drift_reports.append(report)
        
        return drift_reports
    
    def monitor_model_performance(self, predictions: List[int], 
                                 actuals: List[int]) -> Dict:
        """Monitorea performance del modelo con labels reales"""
        if len(predictions) != len(actuals):
            return {}
        
        # Calcular métricas
        correct = sum(p == a for p, a in zip(predictions, actuals))
        accuracy = correct / len(predictions)
        
        # Detección de degradación
        baseline_accuracy = 0.85
        degradation = baseline_accuracy - accuracy
        
        return {
            'accuracy': accuracy,
            'baseline_accuracy': baseline_accuracy,
            'degradation': degradation,
            'alert': degradation > 0.05
        }
    
    def generate_dashboard_metrics(self) -> Dict:
        """Genera métricas para dashboard"""
        return {
            'health_score': self._calculate_health_score(),
            'metrics': self.current_metrics,
            'status': self._get_overall_status()
        }
    
    def _calculate_drift(self, baseline: Dict, production: List[float]) -> float:
        """Calcula drift score (simplificado)"""
        # En producción: usar KS test, PSI, etc.
        baseline_mean = baseline.get('mean', 0)
        prod_mean = sum(production) / len(production) if production else 0
        
        # Normalizar diferencia
        diff = abs(prod_mean - baseline_mean) / (baseline_mean + 1e-6)
        return min(diff, 1.0)
    
    def _calculate_health_score(self) -> float:
        """Calcula score de salud del modelo"""
        score = 100.0
        
        # Penalizar por latencia alta
        if self.current_metrics['latency_p99_ms'] > 100:
            score -= 20
        
        # Penalizar por error rate
        if self.current_metrics['error_rate'] > 0.01:
            score -= 30
        
        return max(score, 0)
    
    def _get_overall_status(self) -> str:
        """Estado general del sistema"""
        health = self._calculate_health_score()
        if health > 80:
            return "healthy"
        elif health > 50:
            return "degraded"
        else:
            return "critical"

# Demo
print("=" * 70)
print("DEMO: ML Observability")
print("=" * 70)

obs = MLObservability()

# Establecer baseline
print("\n📊 Estableciendo baseline de entrenamiento\n")
obs.set_training_baseline({
    'transaction_amount': {'mean': 150.0, 'std': 50.0},
    'user_age': {'mean': 35.0, 'std': 10.0},
    'purchase_frequency': {'mean': 5.0, 'std': 2.0}
})

# Simular datos de producción con drift
print("\n🔍 Detectando drift en producción\n")
production_data = {
    'transaction_amount': [random.uniform(100, 250) for _ in range(100)],  # drift!
    'user_age': [random.uniform(30, 40) for _ in range(100)],  # sin drift
    'purchase_frequency': [random.uniform(4, 6) for _ in range(100)]  # sin drift
}

drift_reports = obs.detect_data_drift(production_data)

for report in drift_reports:
    status = "🚨 DRIFT DETECTADO" if report.is_drifted else "✓ OK"
    print(f"{status} - {report.feature_name}: drift_score={report.drift_score:.3f}")

# Monitorear performance
print("\n📈 Monitoreando performance del modelo\n")
predictions = [random.randint(0, 1) for _ in range(100)]
actuals = [random.randint(0, 1) for _ in range(100)]

perf_metrics = obs.monitor_model_performance(predictions, actuals)
print(f"Accuracy actual: {perf_metrics['accuracy']:.2%}")
print(f"Baseline: {perf_metrics['baseline_accuracy']:.2%}")
print(f"Degradación: {perf_metrics['degradation']:.2%}")

if perf_metrics['alert']:
    print("\n🚨 ALERTA: Degradación significativa detectada")
    print("   Acciones recomendadas:")
    print("   1. Revisar drift en features")
    print("   2. Considerar reentrenamiento del modelo")
    print("   3. Validar calidad de datos recientes")

## 6. Conclusión y mejores prácticas

### Checklist para ML Platform empresarial

#### ✅ Plataforma y herramientas
- [ ] Feature Store centralizado (offline + online)
- [ ] Model Registry con versionado y metadatos
- [ ] Pipelines automatizados de training
- [ ] APIs de inferencia escalables
- [ ] Observabilidad completa (4 capas)

#### ✅ Gobernanza
- [ ] Linaje completo (data → features → model → predictions)
- [ ] Aprobaciones para modelos en producción
- [ ] Control de acceso por equipos
- [ ] Auditoría de decisiones del modelo
- [ ] Compliance con regulaciones (GDPR, etc.)

#### ✅ Operaciones
- [ ] Estrategias de deployment (canary, A/B, shadow)
- [ ] Monitoreo de drift (data + model)
- [ ] Alertas automáticas
- [ ] Rollback automatizado
- [ ] Cost management y optimización

#### ✅ Equipos y procesos
- [ ] Self-service para data scientists
- [ ] Documentación de modelos y features
- [ ] SLAs definidos para inferencia
- [ ] Procesos de incident response
- [ ] Cultura de experimentación

### ROI esperado

Una ML Platform bien implementada genera:
- **70% reducción** en tiempo de deployment de modelos
- **50% menos** costo de infraestructura (compartida)
- **3x más** experimentos por data scientist
- **90% menos** incidentes en producción

### Próximos pasos recomendados

1. **Implementar** Feature Store como base
2. **Estandarizar** pipelines de training
3. **Establecer** Model Registry centralizado
4. **Desplegar** observabilidad completa
5. **Automatizar** deployment con CI/CD
6. **Escalar** a múltiples equipos

---

**Referencias**:
- "Machine Learning Design Patterns" - Lakshmanan et al.
- "Reliable Machine Learning" - Breck, Polyzotis, et al.
- "Building Machine Learning Pipelines" - Hapke & Nelson
- MLOps Community: mlops.community